# 퍼셉트론

In [ ]:
# 퍼셉트론(인공 신경망의 최소 단위)
# 입력(feature)과 가중치를 내적해서 bias를 더한 후,
# activation 함수(비선형 함수)를 통과시켜 결과값을 도출
# activate(x1*w1 + x2*w2 + ... + xn*wn + b)

# activation의 역할: 층을 쌓아도 선형으로 뭉개지지 않게 "비선형성"을 부여하는 것
# (분류/회귀 여부는 activation이 아니라 출력층 activation 종류 + loss 함수가 결정)
 
# 다층 퍼셉트론: 퍼셉트론(내적+bias+activation)의 출력을
# 다음 층의 새로운 weight로 다시 조합해서, 
# 그 결과를 또 다른 퍼셉트론(activation)에 통과시키는 걸 n번 반복하는 구조

# 히든층(입력을 출력으로 변환하는 복잡한 함수 f(x)를 만드는 부분
#  - 노드/층이 많을수록 더 복잡한 함수 표현 가능
#  - 이 함수가 분류에서는 경계선, 회귀에서는 곡선으로 나타남
#  - 회귀/분류 여부는 히든층이 아니라 출력층(activation)과 loss가 결정)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# 데이터 로드 (이미 익숙한 데이터)
iris = load_iris()
X, y = iris.data, iris.target   # X: (150, 4) - 꽃잎/꽃받침 길이너비 4개 feature
                                # y: (150,)  - 품종 0,1,2 세 클래스

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---- 여기가 핵심: 구조를 결정하는 부분 ----
clf = MLPClassifier(
  hidden_layer_sizes=(10, 10),
  # 히든층 2개, 각 층에 노드(퍼셉트론) 10개씩 병렬로 배치
  # 각 노드는 학습을 통해 weight/bias를 스스로 찾아냄
  # (AND/OR처럼 미리 정해진 역할이 있는 게 아니라, 자유롭게 학습됨)
  max_iter=1000,                 # 학습 반복 횟수 (역전파를 몇 번 돌릴지)
  random_state=42
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("정확도:", accuracy_score(y_test, y_pred))

정확도: 0.9333333333333333


# tensorflow
# Keras

In [ ]:
# TensorFlow: 텐서 연산 + 자동 미분을 GPU에서 처리하는 딥러닝 엔진
# Keras: 그 위에서 Sequential + Dense + compile + fit로 신경망을 쉽게 짤 수 있게 해주는 고수준 인터페이스

## 딥러닝 기초 - PyTorch에서 재확인할 개념 목록
(주의: 아래는 TF/Keras 코드로 훑기만 했고, 직접 검증은 안 된 상태. 
PyTorch에서 마주치면 아래 요약을 출발점 삼아 다시 손계산/예시로 확인할 것.)

### Optimizer 세부 종류: loss를 보고 gradient가 나왔을 때 순수 경사하강법보다 더 빠르고 안정적이게 가중치를 조절하기 위해서 쓰는 기법
- **문제 상황**: 기본 경사하강법(SGD)은 매 스텝 "지금 gradient만" 보고 weight를 옮김. 이게 loss 표면이 울퉁불퉁하면 이리저리 흔들리며 느리게 수렴하는 문제가 있음
- **Momentum**: "이전에 이동하던 방향"을 어느 정도 유지하면서 이동. 공을 굴리면 관성 때문에 방향을 급하게 안 바꾸는 것과 비슷 → 흔들림이 줄고 더 빨리 수렴
- **Adagrad**: weight마다 "지금까지 얼마나 많이 움직였는지"를 누적해서, 많이 움직인 weight는 학습률을 점점 줄임 (자주 바뀌던 걸 안정시키는 것). 단점: 계속 누적되기만 해서, 오래 학습하면 학습률이 0에 가까워져 더 이상 못 움직임
- **RMSprop**: Adagrad처럼 학습률을 weight마다 다르게 주되, "최근 gradient에 더 가중치"를 둬서 누적이 무한정 커지지 않게 함 → Adagrad의 "학습률이 너무 빨리 죽는" 문제 완화
- **Adam**: momentum(방향의 관성) + RMSprop(weight별 적응적 학습률) 두 아이디어를 합친 것 → 실무에서 별다른 이유 없으면 기본으로 쓰는 optimizer

### CNN — 이미지처럼 "공간적으로 이웃한 데이터"에서 패턴을 뽑아내는 신경망 구조
- **왜 필요한가**: MLP(Dense만 쓰면)는 이미지를 그냥 숫자 나열로 취급해서, 픽셀들이 서로 이웃해 있다는 공간 정보를 무시함. 사진 속 눈/코/귀가 어디에 있든 같은 패턴이면 같은 걸로 인식해야 하는데, 이게 안 됨
- **Conv2D**: 작은 필터(예: 3x3)를 이미지 위에서 슬라이딩시키며, 그 부분에 모서리·곡선 같은 국소 패턴이 있는지를 뽑아내는 레이어. 필터가 이미지 전체를 돌아다니니까, 패턴이 어디 있든 인식 가능
- **MaxPool2D**: Conv2D가 뽑은 특징 맵을 축소(다운샘플링)해서, 계산량을 줄이고 "위치가 살짝 달라져도 같은 특징"으로 인식하게 도와줌
- **Dense(일반 레이어, MLP의 기본 부품)와 차이**: Dense는 CNN 전용이 아니라 어디서든 쓰는 범용 레이어. CNN에서도 Conv2D+MaxPool2D로 특징을 다 뽑은 다음, 마지막에 "이 특징들을 보고 어느 클래스인지 판단"하는 부분은 결국 Dense(MLP)가 담당함

### RNN / LSTM / GRU — 문장, 시계열처럼 "순서가 있는 데이터"에서 이전 정보를 기억하며 처리하는 신경망 구조
- **RNN**: 직전 스텝의 결과를 기억해서 순서 있는 데이터(문장, 시계열)를 처리할 수 있다
- **RNN(길게 사용할 시)**: 오래된 정보가 점점 흐려지다가 사라진다 (기울기 소실)
- **LSTM**: 게이트로 "기억할지 버릴지"를 정해서, 오래된 정보도 안 사라지고 유지된다
- **GRU**: LSTM보다 가볍게, 비슷한 효과를 낸다

### 과적합 방지
- **공통 목적**: 모델이 학습 데이터의 세세한 노이즈까지 외워버려서, train은 잘 맞추는데 test는 못 맞추는 상태(과적합)를 막는 것
- **정규화**: weight가 0인 것은 필요없는 parameter라 생각해서 아예 없애버림
- **L1 정규화**: loss에 `weight 절댓값의 합`을 더해서, weight가 크면 벌점을 줌. 학습이 진행되며 일부 weight를 아예 0으로 만들어버림 (불필요한 연결을 끊는 효과, sparse)
- **L2 정규화**: loss에 `weight 제곱의 합`을 더해서, 마찬가지로 weight가 크면 벌점. 다만 0으로 만들기보다 전체적으로 골고루 작게 줄임 (weight decay라고도 부름)
- **Dropout**: 학습 중에 매 스텝마다 일부 노드를 무작위로 꺼버림(0으로 만듦). 특정 노드 몇 개에만 의존하는 걸 막고, 결과적으로 여러 개의 작은 모델을 앙상블하는 것과 비슷한 효과
- **BatchNormalization**: 각 층의 입력값을 매 배치마다 다시 정규화(평균0, 분산1 근처로 맞춤). 학습 중 분포가 계속 흔들리는 걸 막아줘서 → 초기화값에 덜 민감해지고, 학습 속도가 빨라지고, 부수적으로 과적합도 줄어드는 효과

### 가중치 초기화 (He 중심)
- **핵심**: 학습 시작 전 weight 초기값이 너무 크거나 작으면 activation 분포가 망가짐 (너무 크면 극단에 쏠려 기울기 소실, 너무 작으면 중앙에 뭉개져 표현력 손실)
- **He 초기화**: `√(2/입력노드수)`. ReLU 전용 — ReLU가 음수를 다 죽이는 걸 감안해 분산을 보정. **지금 쓰는 activation이 ReLU 위주라, 이게 실무에서 실제로 마주칠 조합**
- **(참고, 우선순위 낮음) Xavier 초기화**: `1/√(입력노드수)`. sigmoid/tanh 전용 — 지금 activation 중심이 ReLU라 당장은 몰라도 됨

### 히든층 & ReLU

- **히든층**: 입력을 받아 복잡한 함수 f(x)를 만드는 부분 (분류=경계선, 회귀=곡선으로 나타남). 회귀/분류 여부는 출력층이 결정, 히든층은 무관
- **activation**: 히든층 계산(w·x+b) 뒤에 항상 붙는 필수 부품. 없으면 층을 쌓아도 선형으로 뭉개짐
- **ReLU**: weight를 어떻게 고쳐야 하는지의 신호(gradient)를 sigmoid보다 더 잘 전달
- **주의**: ReLU가 표현력을 높이는 게 아니라, 히든층의 표현력을 "실제로 학습 가능하게" 해주는 것
- **한계**: 음수 구간은 ReLU도 기울기 0 (dying ReLU). sigmoid보단 낫지만 완벽하진 않음
- **He 초기화**: ReLU가 절반(음수)을 죽이는 걸 감안해 분산을 보정(`√(2/n)`). 그래서 ReLU와 항상 세트로 씀

### 텍스트 전처리
- **원-핫 인코딩**: 각 단어가 독립된 벡터가 되어, 숫자 크기로 오인되는 걸 막을 수 있다
- **Word2Vec**: 의미가 비슷한 단어끼리 벡터 공간에서 가까워진다
- **Embedding 레이어**: Word2Vec 같은 단어→벡터 변환을, 신경망 학습 도중 자동으로 함께 배우게 할 수 있다

In [2]:
"""
=====================================================
예제 1. 기본 MLP 학습 파이프라인 (Keras)
- Sequential, Dense, activation, compile, fit 흐름
=====================================================
"""
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = tf.keras.Sequential([
  tf.keras.layers.Dense(16, activation='relu'),   # 히든층: activation 필수 (비선형성)
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dense(3, activation='softmax')  # 출력층: 다중분류라 softmax
])

# loss/optimizer/metrics를 미리 정해두는 단계 (아직 학습 안 함)
model.compile(
  loss='sparse_categorical_crossentropy',  # 라벨이 정수(0,1,2)일 때
  optimizer='adam',                          # 실무 기본값 (momentum + 적응적 학습률)
  metrics=['accuracy']
)

model.summary()

# epochs: 전체 데이터를 몇 바퀴 도는지
# batch_size: 한 번 업데이트에 데이터를 얼마나 보는지
history = model.fit(
  X_train, y_train,
  epochs=50,
  batch_size=16,
  validation_data=(X_test, y_test),
  verbose=0
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("정확도:", acc)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

정확도: 1.0


In [ ]:
"""
=====================================================
예제 2. 과적합 방지 (L1/L2 정규화 + Dropout + BatchNorm)
- 실무에서 거의 항상 같이 쓰는 3종 세트
=====================================================
"""
import tensorflow as tf

model = tf.keras.Sequential([
  tf.keras.layers.Dense(
    128,
    kernel_regularizer=tf.keras.regularizers.l2(0.001)  # L2: weight 크기 억제
  ),
  tf.keras.layers.BatchNormalization(),   # Dense 직후, Activation 직전
  tf.keras.layers.Activation('relu'),

  tf.keras.layers.Dense(
    64,
    kernel_regularizer=tf.keras.regularizers.l2(0.001)
  ),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.Activation('relu'),

  tf.keras.layers.Dropout(0.3),  # 마지막 히든층-출력층 사이 한 곳만

  tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
"""
=====================================================
예제 3. 가중치 초기화 지정 (He / Xavier)
- relu엔 He, sigmoid/tanh엔 Xavier(Glorot)가 실무 기본값
=====================================================
"""
import tensorflow as tf

model = tf.keras.Sequential([
  tf.keras.layers.Dense(
    128, activation='relu',
    kernel_initializer='he_normal'        # relu용
  ),
  tf.keras.layers.Dense(
    64, activation='tanh',
    kernel_initializer='glorot_normal'    # Xavier의 다른 이름, tanh/sigmoid용
  ),
  tf.keras.layers.Dense(10, activation='softmax')
])

# 참고: activation='relu'만 쓰면 Keras가 기본적으로 glorot_uniform을 씀
# he_normal을 명시해주는 게 relu에는 더 적합

In [ ]:
"""
=====================================================
예제 4. CNN (이미지 분류) - Conv2D, MaxPool2D, BatchNorm
=====================================================
"""
import tensorflow as tf
import numpy as np

mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = np.expand_dims(x_train / 255.0, axis=-1)  # (N,28,28) -> (N,28,28,1) 채널 추가
x_test = np.expand_dims(x_test / 255.0, axis=-1)

model = tf.keras.Sequential([
  tf.keras.layers.Conv2D(32, (3, 3), padding='same', input_shape=(28, 28, 1)),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.Activation('relu'),
  tf.keras.layers.MaxPool2D(pool_size=(2, 2)),

  tf.keras.layers.Conv2D(64, (3, 3), padding='same'),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.Activation('relu'),
  tf.keras.layers.MaxPool2D(pool_size=(2, 2)),

  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(64, activation='relu'),
  tf.keras.layers.Dropout(0.3),
  tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(x_train, y_train, epochs=5, batch_size=128,
           validation_data=(x_test, y_test), verbose=0)

In [ ]:
"""
=====================================================
예제 5. RNN 계열 (시계열/텍스트) - Embedding, LSTM
- 예지보전 센서 시계열에도 구조적으로 재사용 가능한 패턴
  (Embedding 대신 센서값 그대로, 시퀀스 형태만 유지하면 됨)
=====================================================
"""
import tensorflow as tf

vocab_size = 1000
max_len = 300

# 텍스트 예시 (Embedding 사용)
text_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 32, input_length=max_len),  # 정수 인덱스 -> 학습가능한 벡터
    tf.keras.layers.LSTM(32),        # SimpleRNN보다 장기 의존성 처리 우수
    tf.keras.layers.Dense(1, activation='sigmoid')  # 이진분류
])
text_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])


# 시계열 센서 데이터 예시 (Embedding 없이, 수치 시퀀스 그대로)
# 입력 shape: (샘플 수, 타임스텝 수, 센서 개수)
n_timesteps = 50
n_sensors = 5

sensor_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64, input_shape=(n_timesteps, n_sensors), return_sequences=False),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # 예: 고장(1)/정상(0)
])
sensor_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
"""
=====================================================
예제 6. 원-핫 인코딩된 라벨 사용 시 loss 차이
- to_categorical 쓸 때는 categorical_crossentropy
- 정수 라벨 그대로 쓸 때는 sparse_categorical_crossentropy
=====================================================
"""
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

y_train_int = [0, 1, 2, 1, 0]           # 정수 라벨
y_train_onehot = to_categorical(y_train_int, num_classes=3)  # [[1,0,0],[0,1,0],...]

# 정수 라벨 그대로 쓸 경우
model_a = tf.keras.Sequential([tf.keras.layers.Dense(3, activation='softmax')])
model_a.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

# 원-핫 라벨 쓸 경우
model_b = tf.keras.Sequential([tf.keras.layers.Dense(3, activation='softmax')])
model_b.compile(loss='categorical_crossentropy', optimizer='adam')

성공
